In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Qwen2.5-VL LoRA fine-tune (optional)

Minimal LoRA scaffold. You may need to adjust the prompt and masking for your model.


In [2]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import AutoProcessor, AutoModelForCausalLM, Trainer, TrainingArguments
try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

try:
    import bitsandbytes as bnb  # noqa: F401
    HAS_BNB = True
except Exception:
    HAS_BNB = False
try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False


2026-02-04 11:37:58.787179: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-04 11:37:58.787215: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-04 11:37:58.788321: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-04 11:37:58.794852: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-04 11:37:59.652164: W tensorflow/compiler/tf2

In [3]:
from pathlib import Path
import os

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
sys.path.append(str(ROOT))

from common import (
    find_long_table,
    load_long_table,
    normalize_answer,
)

DATA_PATH = find_long_table(ROOT)
OUT_DIR = ROOT / "3_modern_vlm" / "out" / "06_qwen2_5_vl_lora_finetune"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = ROOT / "3_modern_vlm" / "results" / "06_qwen2_5_vl_lora_finetune"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.environ.get("VLM_MODEL_NAME", "Qwen/Qwen2.5-VL-7B-Instruct")
BATCH_SIZE = 1
EPOCHS = 1
LR = 2e-4
MAX_TRAIN_SAMPLES = int(os.environ.get("MAX_TRAIN_SAMPLES", "0")) or None
MAX_TEXT_LEN = 128
USE_4BIT = True
USE_8BIT = False
DEVICE_MAP = "auto"  # set to None to disable device map
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("LoRA fine-tuning requires a CUDA GPU. Set up GPU runtime and rerun.")


In [4]:
if not HAS_PEFT:
    raise ImportError("peft is required for LoRA fine-tuning. Install with `pip install peft`. ")

long_df = load_long_table(DATA_PATH)
train_df = long_df[long_df["split"] == "train"].copy()
if MAX_TRAIN_SAMPLES:
    train_df = train_df.head(MAX_TRAIN_SAMPLES)


In [5]:
if (USE_4BIT or USE_8BIT) and not HAS_BNB:
    raise ImportError(
        "bitsandbytes is required for 4/8-bit quantization. "
        "Install with `pip install bitsandbytes` or set USE_4BIT/USE_8BIT to False."
    )


quant_config = None
if DEVICE == "cuda" and BitsAndBytesConfig is not None:
    if USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
    elif USE_8BIT:
        quant_config = BitsAndBytesConfig(load_in_8bit=True)

load_kwargs = {
    "torch_dtype": torch.float16 if DEVICE == "cuda" else torch.float32,
    "trust_remote_code": True,
}
if DEVICE == "cuda" and DEVICE_MAP:
    load_kwargs["device_map"] = DEVICE_MAP
if quant_config is not None:
    load_kwargs["quantization_config"] = quant_config

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

if AutoModelForVision2Seq is not None:
    try:
        base_model = AutoModelForVision2Seq.from_pretrained(
            MODEL_NAME,
            **load_kwargs,
        )
    except Exception:
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            **load_kwargs,
        )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        trust_remote_code=True,
    )

base_model = prepare_model_for_kbit_training(base_model)

# Infer LoRA target modules (common Qwen/LLaMA blocks)
_candidate_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
_found = set()
for name, _ in base_model.named_modules():
    for c in _candidate_modules:
        if name.endswith(c):
            _found.add(c)
_target_modules = sorted(_found) if _found else _candidate_modules

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=_target_modules,
)
model = get_peft_model(base_model, lora_config)
if DEVICE_MAP is None:
    model.to(DEVICE)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
# Prompt helpers

def format_prompt(question: str) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    return f"USER: <image>Question: {q}ASSISTANT:"


In [7]:
class VQALoRADataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        question = row["question_text"]
        answer = normalize_answer(row["answer_norm"])

        prompt = format_prompt(question)
        full_text = prompt + " " + answer

        full = processor(images=image, text=full_text, return_tensors="pt", padding=False, truncation=False)
        prompt_only = processor(images=image, text=prompt, return_tensors="pt", padding=False, truncation=False)

        input_ids = full["input_ids"].squeeze(0)
        attention_mask = full["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        prompt_len = prompt_only["input_ids"].shape[1]
        labels[:prompt_len] = -100

        item = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

        # Keep multimodal fields (e.g., pixel_values, image_grid_thw) as returned by the processor
        for k, v in full.items():
            if k in ("input_ids", "attention_mask"):
                continue
            if isinstance(v, torch.Tensor):
                item[k] = v

        return item


# Dynamic padding for variable-length multimodal batches
def collate_fn(features):
    input_ids = [f["input_ids"] for f in features]
    attention_mask = [f["attention_mask"] for f in features]
    labels = [f["labels"] for f in features]

    batch = processor.tokenizer.pad(
        {"input_ids": input_ids, "attention_mask": attention_mask},
        padding=True,
        return_tensors="pt",
    )

    max_len = batch["input_ids"].shape[1]
    labels_padded = torch.full((len(labels), max_len), -100, dtype=labels[0].dtype)
    for i, l in enumerate(labels):
        labels_padded[i, : l.shape[0]] = l
    batch["labels"] = labels_padded

    visual_concat_keys = {
        "pixel_values",
        "image_grid_thw",
        "pixel_values_videos",
        "video_grid_thw",
        "second_per_grid_ts",
    }

    extra_keys = [k for k in features[0].keys() if k not in ("input_ids", "attention_mask", "labels")]
    for k in extra_keys:
        vals = [f[k] for f in features]
        if isinstance(vals[0], torch.Tensor):
            if k in visual_concat_keys:
                batch[k] = torch.cat(vals, dim=0)
            else:
                batch[k] = torch.stack(vals)
        else:
            batch[k] = vals

    return batch

train_ds = VQALoRADataset(train_df)


In [8]:
args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [9]:
# Train
trainer.train()


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1154: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradi

Step,Training Loss
20,9.901700


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.50 GiB. GPU 0 has a total capacity of 15.47 GiB of which 1.42 GiB is free. Including non-PyTorch memory, this process has 13.38 GiB memory in use. Of the allocated memory 12.29 GiB is allocated by PyTorch, and 842.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Save LoRA adapter
model.save_pretrained(OUT_DIR / "lora_adapter")

# Save lightweight results
summary = {
    "model_name": MODEL_NAME,
    "run": "06_qwen2_5_vl_lora_finetune",
    "num_train_samples": int(len(train_df)),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "use_4bit": USE_4BIT,
    "use_8bit": USE_8BIT,
    "device_map": DEVICE_MAP,
}
with open(RESULTS_DIR / "train_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

try:
    trainer.state.save_to_json(str(RESULTS_DIR / "trainer_state.json"))
except Exception as e:
    print(f"Warning: could not save trainer_state.json: {e}")

if getattr(trainer.state, "log_history", None) is not None:
    with open(RESULTS_DIR / "log_history.json", "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

OUT_DIR
RESULTS_DIR
